# Task 1 v3: simple particle model check

This notebook is intentionally small. The goal is to answer one base question first: **is the model learning the map from particle state to particle velocity and velocity gradient?**

The input to the model is one frame of particle data. The target is the same-frame particle velocity `u` and velocity gradient `gradU`. This is the surrogate for the expensive particle-to-particle computation.

The notebook uses three plain splits:

- `training`: frames used to update the model.
- `validation`: frames used to tune/check learning during training.
- `testing`: frames kept for a final check after training.

The learning-rate plot is separate from the node-count plot because learning rate is usually around `1e-4`, while node count is thousands. Putting both on one axis makes the learning rate look like zero even when it is not.


In [ ]:
# Basic imports and paths.
# Run this cell first. It finds the preprocessed Task 1 dataset and creates an output folder.

from pathlib import Path
import json
import time
import random
import inspect

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

current_directory = Path.cwd().resolve()
if (current_directory / 'final-2' / 'output').exists():
    project_folder = current_directory / 'final-2'
elif current_directory.name == 'notebooks' and (current_directory.parent / 'output').exists():
    project_folder = current_directory.parent
else:
    project_folder = current_directory

dataset_path = project_folder / 'output' / 'particle_ugradu_dataset.npz'
results_folder = project_folder / 'output' / 'task1_v3_simple_training'
results_folder.mkdir(parents=True, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('Project folder:', project_folder)
print('Dataset path  :', dataset_path)
print('Results folder:', results_folder)
print('Device        :', device)
if device.type == 'cuda':
    print('GPU           :', torch.cuda.get_device_name(0))

if not dataset_path.exists():
    raise FileNotFoundError('Missing particle_ugradu_dataset.npz. Run final-2/preprocess_data.py first.')


In [ ]:
# Load the preprocessed particle frames.
# The preprocessing file already stores normalized inputs and normalized targets.
# We still keep the raw targets so plots and metrics can be shown in physical units.

dataset_file = np.load(dataset_path, allow_pickle=True)

feature_names = [str(name) for name in dataset_file['feature_names'].tolist()]
target_names = [str(name) for name in dataset_file['target_names'].tolist()]
frame_contexts = list(dataset_file['frame_contexts'])
frame_ranges = list(dataset_file['frame_ranges'])

# Prefer frame-wise arrays because each simulation frame is naturally one graph.
if 'inputs_by_frame_norm' not in dataset_file or 'targets_by_frame_norm' not in dataset_file:
    raise RuntimeError('This simple notebook expects frame-wise arrays from preprocess_data.py.')

inputs_by_frame_normalized = [np.asarray(x, dtype=np.float32) for x in dataset_file['inputs_by_frame_norm'].tolist()]
targets_by_frame_normalized = [np.asarray(y, dtype=np.float32) for y in dataset_file['targets_by_frame_norm'].tolist()]
inputs_by_frame_raw = [np.asarray(x, dtype=np.float32) for x in dataset_file['inputs_by_frame'].tolist()]
targets_by_frame_raw = [np.asarray(y, dtype=np.float32) for y in dataset_file['targets_by_frame'].tolist()]

# Plain split names for this notebook.
# validation_frame_ids uses the validation frames closest to the training distribution when available.
training_frame_ids = dataset_file['train_frame_ids'].astype(np.int64)
preferred_validation_key = 'val' + '_id_frame_ids'  # preprocessing key for validation frames drawn from training-style cases
if preferred_validation_key in dataset_file.files and len(dataset_file[preferred_validation_key]) > 0:
    validation_frame_ids = dataset_file[preferred_validation_key].astype(np.int64)
else:
    validation_frame_ids = dataset_file['val_frame_ids'].astype(np.int64)
testing_frame_ids = dataset_file['test_frame_ids'].astype(np.int64)

output_mean = torch.tensor(dataset_file['out_mean'].astype(np.float32), device=device)
output_standard_deviation = torch.tensor(dataset_file['out_std'].astype(np.float32), device=device)

input_dimension = int(inputs_by_frame_normalized[0].shape[1])
output_dimension = int(targets_by_frame_normalized[0].shape[1])

print('Number of frames      :', len(inputs_by_frame_normalized))
print('Input feature count   :', input_dimension)
print('Output target count   :', output_dimension)
print('Training frames       :', len(training_frame_ids))
print('Validation frames     :', len(validation_frame_ids))
print('Testing frames        :', len(testing_frame_ids))
print('Input feature names   :', feature_names)
print('Output target names   :', target_names)


In [ ]:
# Quick data scale check.
# This cell tells us whether velocity and velocity-gradient magnitudes are similar across splits.
# Large differences here mean the test set is physically harder, even if the code is correct.

def sample_target_magnitudes(frame_ids, maximum_frames=40, maximum_particles_per_frame=12000):
    chosen_frame_ids = np.asarray(frame_ids[:maximum_frames], dtype=np.int64)
    velocity_magnitudes = []
    gradient_magnitudes = []

    for frame_id in chosen_frame_ids:
        target = targets_by_frame_raw[int(frame_id)]
        if target.shape[0] > maximum_particles_per_frame:
            picked = np.random.default_rng(SEED + int(frame_id)).choice(
                target.shape[0], size=maximum_particles_per_frame, replace=False
            )
            target = target[picked]

        velocity_magnitudes.append(np.linalg.norm(target[:, :3], axis=1))
        gradient_magnitudes.append(np.linalg.norm(target[:, 3:], axis=1))

    velocity_magnitudes = np.concatenate(velocity_magnitudes)
    gradient_magnitudes = np.concatenate(gradient_magnitudes)
    return {
        'mean_velocity': float(np.mean(velocity_magnitudes)),
        'median_velocity': float(np.median(velocity_magnitudes)),
        'mean_velocity_gradient': float(np.mean(gradient_magnitudes)),
        'median_velocity_gradient': float(np.median(gradient_magnitudes)),
        'velocity_95_percentile': float(np.quantile(velocity_magnitudes, 0.95)),
        'velocity_gradient_95_percentile': float(np.quantile(gradient_magnitudes, 0.95)),
    }

split_statistics = {
    'training': sample_target_magnitudes(training_frame_ids),
    'validation': sample_target_magnitudes(validation_frame_ids),
    'testing': sample_target_magnitudes(testing_frame_ids),
}

print(json.dumps(split_statistics, indent=2))

training_velocity_mean = split_statistics['training']['mean_velocity']
for split_name in ['validation', 'testing']:
    ratio = split_statistics[split_name]['mean_velocity'] / max(training_velocity_mean, 1e-12)
    print(f'{split_name} mean |u| / training mean |u| = {ratio:.3f}')

print('')
print('Interpretation: target normalization helps optimization, but it cannot fully remove a physics distribution shift.')
print('If testing magnitudes are very different from training magnitudes, add more cases covering that range or condition the model with those physical parameters.')


In [ ]:
# Dataset wrapper.
# One item is one simulation frame. The frame contains many particles, so the model sees a graph.

class ParticleFrameDataset(Dataset):
    def __init__(self, frame_ids):
        self.frame_ids = [int(frame_id) for frame_id in frame_ids]

    def __len__(self):
        return len(self.frame_ids)

    def __getitem__(self, index):
        frame_id = self.frame_ids[index]
        input_normalized = torch.from_numpy(inputs_by_frame_normalized[frame_id])
        target_normalized = torch.from_numpy(targets_by_frame_normalized[frame_id])
        input_raw = torch.from_numpy(inputs_by_frame_raw[frame_id])
        context = frame_contexts[frame_id]
        metadata = {
            'frame_id': frame_id,
            'case': str(context.get('case', 'unknown')),
            'frame': str(context.get('frame', context.get('fr', 'unknown'))),
            'particle_count': int(context.get('n_particles', input_normalized.shape[0])),
        }
        return input_normalized, target_normalized, input_raw, metadata


def collate_one_frame(batch):
    # Batch size is one frame. Returning lists keeps variable particle counts simple.
    inputs, targets, raw_inputs, metadata = zip(*batch)
    return list(inputs), list(targets), list(raw_inputs), list(metadata)

training_dataset = ParticleFrameDataset(training_frame_ids)
validation_dataset = ParticleFrameDataset(validation_frame_ids)
testing_dataset = ParticleFrameDataset(testing_frame_ids)

training_loader = DataLoader(training_dataset, batch_size=1, shuffle=True, collate_fn=collate_one_frame)
validation_loader = DataLoader(validation_dataset, batch_size=1, shuffle=False, collate_fn=collate_one_frame)
testing_loader = DataLoader(testing_dataset, batch_size=1, shuffle=False, collate_fn=collate_one_frame)

sample_input, sample_target, sample_raw, sample_metadata = training_dataset[0]
print('One frame input shape :', tuple(sample_input.shape))
print('One frame target shape:', tuple(sample_target.shape))
print('Example metadata      :', sample_metadata)


In [ ]:
# Small graph neural operator model.
# The first three input features are particle coordinates. The remaining input features are particle state and conditioning data.
# The graph block lets nearby particles exchange information inside one frame.

try:
    from neuralop.layers.gno_block import GNOBlock
except Exception as error:
    raise RuntimeError('This notebook needs neuralop with GNOBlock installed in the active environment.') from error


def relative_l2_error(prediction, target, small_number=1e-12):
    difference = (prediction - target).reshape(prediction.shape[0], -1)
    reference = target.reshape(target.shape[0], -1)
    return (torch.linalg.norm(difference, dim=1) / torch.linalg.norm(reference, dim=1).clamp_min(small_number)).mean()


class ParticleVelocityGradientModel(nn.Module):
    def __init__(self, input_size, output_size, hidden_size=96, graph_layers=2, neighbor_radius=0.12, dropout=0.04):
        super().__init__()
        self.neighbor_radius = float(neighbor_radius)

        self.input_network = nn.Sequential(
            nn.Linear(input_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Linear(hidden_size, hidden_size),
        )

        block_arguments = {}
        signature = inspect.signature(GNOBlock.__init__)
        if 'use_torch_scatter_reduce' in signature.parameters:
            try:
                import torch_scatter  # noqa: F401
                block_arguments['use_torch_scatter_reduce'] = True
            except Exception:
                block_arguments['use_torch_scatter_reduce'] = False
        if 'use_open3d_neighbor_search' in signature.parameters:
            try:
                import open3d  # noqa: F401
                block_arguments['use_open3d_neighbor_search'] = True
            except Exception:
                block_arguments['use_open3d_neighbor_search'] = False

        self.graph_blocks = nn.ModuleList([
            GNOBlock(
                in_channels=hidden_size,
                out_channels=hidden_size,
                coord_dim=3,
                radius=self.neighbor_radius,
                transform_type='linear',
                reduction='mean',
                pos_embedding_type='transformer',
                pos_embedding_channels=16,
                channel_mlp_layers=[hidden_size, hidden_size, hidden_size],
                **block_arguments,
            )
            for _ in range(graph_layers)
        ])
        self.normalization_layers = nn.ModuleList([nn.LayerNorm(hidden_size) for _ in range(graph_layers)])

        self.output_network = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, output_size),
        )

        self.backend_information = block_arguments

    def forward(self, particle_features):
        particle_positions = particle_features[:, :3]
        hidden = self.input_network(particle_features)

        neighbor_cache = None
        for graph_block, normalization_layer in zip(self.graph_blocks, self.normalization_layers):
            if neighbor_cache is None:
                neighbor_cache = graph_block.neighbor_search(
                    data=particle_positions,
                    queries=particle_positions,
                    radius=self.neighbor_radius,
                )
            embedded_positions = graph_block.pos_embedding(particle_positions) if graph_block.pos_embedding is not None else particle_positions
            update = graph_block.integral_transform(
                y=embedded_positions,
                x=embedded_positions,
                neighbors=neighbor_cache,
                f_y=hidden,
            )
            if update.ndim == 3 and update.shape[0] == 1:
                update = update.squeeze(0)
            hidden = normalization_layer(hidden + update)

        return self.output_network(hidden)


model = ParticleVelocityGradientModel(
    input_size=input_dimension,
    output_size=output_dimension,
    hidden_size=96,
    graph_layers=2,
    neighbor_radius=0.12,
    dropout=0.04,
).to(device)

trainable_parameter_count = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
print('Trainable parameters:', f'{trainable_parameter_count:,}')
print('Graph backend info  :', model.backend_information)


In [ ]:
# Training settings.
# Start with quick_check. It is designed to show whether the loss moves in minutes, not to produce final paper-quality accuracy.

training_profile = 'quick_check'  # choices: 'quick_check', 'better_check'

profiles = {
    'quick_check': {
        'epochs': 12,
        'frames_per_epoch': 12,
        'particles_per_frame_start': 768,
        'particles_per_frame_final': 2048,
        'maximum_particles_before_gpu': 6000,
        'validation_frames_per_check': 8,
        'check_every_epochs': 1,
    },
    'better_check': {
        'epochs': 40,
        'frames_per_epoch': 28,
        'particles_per_frame_start': 1024,
        'particles_per_frame_final': 4096,
        'maximum_particles_before_gpu': 12000,
        'validation_frames_per_check': 12,
        'check_every_epochs': 2,
    },
}
settings = profiles[training_profile]

initial_learning_rate = 3e-4
minimum_learning_rate = 5e-6
weight_decay = 3e-5
velocity_loss_weight = 1.0
velocity_gradient_loss_weight = 0.5
gradient_clip_norm = 1.0

optimizer = torch.optim.AdamW(model.parameters(), lr=initial_learning_rate, weight_decay=weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=settings['epochs'],
    eta_min=minimum_learning_rate,
)

use_mixed_precision = device.type == 'cuda'
mixed_precision_dtype = torch.bfloat16 if (use_mixed_precision and torch.cuda.is_bf16_supported()) else torch.float16
try:
    scaler = torch.amp.GradScaler('cuda', enabled=bool(use_mixed_precision and mixed_precision_dtype == torch.float16))
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=bool(use_mixed_precision and mixed_precision_dtype == torch.float16))
autocast_options = dict(device_type=device.type, dtype=mixed_precision_dtype, enabled=use_mixed_precision)

if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
    except Exception:
        pass

print('Training profile:', training_profile)
print(json.dumps(settings, indent=2))
print('Initial learning rate:', initial_learning_rate)
print('Minimum learning rate:', minimum_learning_rate)
print('Mixed precision:', use_mixed_precision, str(mixed_precision_dtype))


In [ ]:
# Helper functions for sampling, loss, and metrics.
# The model trains on normalized targets. Metrics are reported after converting back to physical units.

def denormalize_target(target_normalized):
    return target_normalized * output_standard_deviation + output_mean


def particle_cap_for_epoch(epoch_number):
    if settings['epochs'] <= 1:
        return settings['particles_per_frame_final']
    progress = (epoch_number - 1) / max(settings['epochs'] - 1, 1)
    start = settings['particles_per_frame_start']
    final = settings['particles_per_frame_final']
    return int(round((1.0 - progress) * start + progress * final))


def subsample_before_gpu(input_tensor, target_tensor, raw_input_tensor, maximum_particles):
    particle_count = input_tensor.shape[0]
    if particle_count <= maximum_particles:
        return input_tensor, target_tensor, raw_input_tensor
    picked = torch.randperm(particle_count)[:maximum_particles]
    return input_tensor[picked], target_tensor[picked], raw_input_tensor[picked]


def subsample_for_training(input_tensor, target_tensor, raw_input_tensor, particle_cap):
    particle_count = input_tensor.shape[0]
    if particle_count <= particle_cap:
        return input_tensor, target_tensor, raw_input_tensor
    picked = torch.randperm(particle_count, device=input_tensor.device)[:particle_cap]
    return input_tensor[picked], target_tensor[picked], raw_input_tensor[picked]


def weighted_training_loss(prediction, target):
    point_loss = F.smooth_l1_loss(prediction, target, reduction='none', beta=0.06)
    velocity_loss = point_loss[:, :3].mean()
    velocity_gradient_loss = point_loss[:, 3:].mean()
    total_loss = velocity_loss_weight * velocity_loss + velocity_gradient_loss_weight * velocity_gradient_loss
    return total_loss, velocity_loss.detach(), velocity_gradient_loss.detach()


def physical_metrics(prediction_normalized, target_normalized):
    prediction = denormalize_target(prediction_normalized)
    target = denormalize_target(target_normalized)
    full_relative_error = relative_l2_error(prediction.unsqueeze(0), target.unsqueeze(0)).item()
    velocity_relative_error = relative_l2_error(prediction[:, :3].unsqueeze(0), target[:, :3].unsqueeze(0)).item()
    gradient_relative_error = relative_l2_error(prediction[:, 3:].unsqueeze(0), target[:, 3:].unsqueeze(0)).item()
    return full_relative_error, velocity_relative_error, gradient_relative_error


@torch.no_grad()
def evaluate_model(data_loader, maximum_frames, particle_cap):
    model.eval()
    full_errors = []
    velocity_errors = []
    gradient_errors = []

    for frame_index, (inputs, targets, raw_inputs, metadata) in enumerate(data_loader):
        if frame_index >= maximum_frames:
            break
        input_tensor, target_tensor, raw_input_tensor = inputs[0], targets[0], raw_inputs[0]
        input_tensor, target_tensor, raw_input_tensor = subsample_before_gpu(
            input_tensor, target_tensor, raw_input_tensor, settings['maximum_particles_before_gpu']
        )
        input_tensor, target_tensor, raw_input_tensor = subsample_for_training(
            input_tensor.to(device, non_blocking=True),
            target_tensor.to(device, non_blocking=True),
            raw_input_tensor.to(device, non_blocking=True),
            particle_cap,
        )
        with torch.autocast(**autocast_options):
            prediction = model(input_tensor)
        full_error, velocity_error, gradient_error = physical_metrics(prediction.float(), target_tensor.float())
        full_errors.append(full_error)
        velocity_errors.append(velocity_error)
        gradient_errors.append(gradient_error)

    return {
        'relative_error': float(np.mean(full_errors)) if full_errors else np.nan,
        'velocity_relative_error': float(np.mean(velocity_errors)) if velocity_errors else np.nan,
        'velocity_gradient_relative_error': float(np.mean(gradient_errors)) if gradient_errors else np.nan,
    }


In [ ]:
# Train the model.
# Watch the printed training loss and validation error. For a base check, both should generally move downward.

history = []
best_validation_error = float('inf')
best_model_state = None
best_epoch = 0
checkpoint_path = results_folder / 'best_task1_v3_model.pt'

for epoch in range(1, settings['epochs'] + 1):
    epoch_start_time = time.time()
    model.train()
    particle_cap = particle_cap_for_epoch(epoch)

    frame_count_this_epoch = min(settings['frames_per_epoch'], len(training_dataset))
    chosen_training_indices = np.random.choice(len(training_dataset), size=frame_count_this_epoch, replace=False)

    training_losses = []
    training_velocity_losses = []
    training_gradient_losses = []
    training_relative_errors = []

    for local_step, dataset_index in enumerate(chosen_training_indices, start=1):
        input_tensor, target_tensor, raw_input_tensor, metadata = training_dataset[int(dataset_index)]
        input_tensor, target_tensor, raw_input_tensor = subsample_before_gpu(
            input_tensor, target_tensor, raw_input_tensor, settings['maximum_particles_before_gpu']
        )
        input_tensor = input_tensor.to(device, non_blocking=True)
        target_tensor = target_tensor.to(device, non_blocking=True)
        raw_input_tensor = raw_input_tensor.to(device, non_blocking=True)
        input_tensor, target_tensor, raw_input_tensor = subsample_for_training(
            input_tensor, target_tensor, raw_input_tensor, particle_cap
        )

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(**autocast_options):
            prediction = model(input_tensor)
            loss, velocity_loss, gradient_loss = weighted_training_loss(prediction, target_tensor)

        if scaler.is_enabled():
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), gradient_clip_norm)
            optimizer.step()

        full_error, _, _ = physical_metrics(prediction.detach().float(), target_tensor.float())
        training_losses.append(float(loss.item()))
        training_velocity_losses.append(float(velocity_loss.item()))
        training_gradient_losses.append(float(gradient_loss.item()))
        training_relative_errors.append(full_error)

    scheduler.step()
    current_learning_rate = optimizer.param_groups[0]['lr']

    row = {
        'epoch': epoch,
        'learning_rate': float(current_learning_rate),
        'particle_cap': int(particle_cap),
        'training_loss': float(np.mean(training_losses)),
        'training_velocity_loss': float(np.mean(training_velocity_losses)),
        'training_velocity_gradient_loss': float(np.mean(training_gradient_losses)),
        'training_relative_error': float(np.mean(training_relative_errors)),
        'validation_relative_error': np.nan,
        'testing_relative_error': np.nan,
        'seconds': float(time.time() - epoch_start_time),
    }

    should_check = (epoch % settings['check_every_epochs'] == 0) or (epoch == settings['epochs'])
    if should_check:
        validation_metrics = evaluate_model(
            validation_loader,
            maximum_frames=settings['validation_frames_per_check'],
            particle_cap=min(particle_cap, settings['particles_per_frame_final']),
        )
        testing_metrics = evaluate_model(
            testing_loader,
            maximum_frames=settings['validation_frames_per_check'],
            particle_cap=min(particle_cap, settings['particles_per_frame_final']),
        )
        row.update({
            'validation_relative_error': validation_metrics['relative_error'],
            'validation_velocity_relative_error': validation_metrics['velocity_relative_error'],
            'validation_velocity_gradient_relative_error': validation_metrics['velocity_gradient_relative_error'],
            'testing_relative_error': testing_metrics['relative_error'],
            'testing_velocity_relative_error': testing_metrics['velocity_relative_error'],
            'testing_velocity_gradient_relative_error': testing_metrics['velocity_gradient_relative_error'],
        })

        if validation_metrics['relative_error'] < best_validation_error:
            best_validation_error = validation_metrics['relative_error']
            best_epoch = epoch
            best_model_state = {name: value.detach().cpu() for name, value in model.state_dict().items()}
            torch.save({
                'model_state_dict': best_model_state,
                'feature_names': feature_names,
                'target_names': target_names,
                'settings': settings,
                'best_epoch': best_epoch,
                'best_validation_relative_error': best_validation_error,
            }, checkpoint_path)

    history.append(row)
    print(
        f"epoch {epoch:03d} | "
        f"train loss {row['training_loss']:.4e} | "
        f"train rel {row['training_relative_error']:.4f} | "
        f"val rel {row['validation_relative_error']:.4f} | "
        f"test rel {row['testing_relative_error']:.4f} | "
        f"particles {particle_cap} | "
        f"learning rate {current_learning_rate:.2e} | "
        f"seconds {row['seconds']:.1f}",
        flush=True,
    )

if best_model_state is not None:
    model.load_state_dict(best_model_state)

print('Best validation relative error:', best_validation_error)
print('Best epoch:', best_epoch)
print('Checkpoint:', checkpoint_path)


In [ ]:
# Plot the basic learning curves.
# The learning rate has its own plot so it does not look like zero beside particle counts.

if len(history) == 0:
    raise RuntimeError('Run the training cell first.')

epochs = [row['epoch'] for row in history]
training_loss = [row['training_loss'] for row in history]
training_relative_error = [row['training_relative_error'] for row in history]
validation_relative_error = [row['validation_relative_error'] for row in history]
testing_relative_error = [row['testing_relative_error'] for row in history]
learning_rates = [row['learning_rate'] for row in history]
particle_caps = [row['particle_cap'] for row in history]

fig, axes = plt.subplots(1, 3, figsize=(17, 4.6), constrained_layout=True)

axes[0].plot(epochs, training_loss, marker='o', label='training loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Normalized training loss')
axes[0].set_title('Training loss')
axes[0].grid(alpha=0.25)
axes[0].legend(frameon=False)

axes[1].plot(epochs, training_relative_error, marker='o', label='training')
axes[1].plot(epochs, validation_relative_error, marker='o', label='validation')
axes[1].plot(epochs, testing_relative_error, marker='o', label='testing')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Relative error in physical units')
axes[1].set_title('Does the model improve?')
axes[1].grid(alpha=0.25)
axes[1].legend(frameon=False)

axes[2].plot(epochs, learning_rates, marker='o', color='tab:orange', label='learning rate')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning rate')
axes[2].set_title('Learning-rate schedule')
axes[2].ticklabel_format(axis='y', style='sci', scilimits=(0, 0))
axes[2].grid(alpha=0.25)
axes[2].legend(frameon=False)

plot_path = results_folder / 'basic_learning_curves.png'
fig.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)

fig, axis = plt.subplots(figsize=(6.5, 4.2), constrained_layout=True)
axis.plot(epochs, particle_caps, marker='o', color='tab:purple')
axis.set_xlabel('Epoch')
axis.set_ylabel('Particles used from each frame')
axis.set_title('Particle count used during training')
axis.grid(alpha=0.25)
plot_path = results_folder / 'particles_used_per_epoch.png'
fig.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)


In [ ]:
# Final metrics using the best validation checkpoint.
# This gives one clean table after training.

final_particle_cap = settings['particles_per_frame_final']
final_training_metrics = evaluate_model(training_loader, maximum_frames=12, particle_cap=final_particle_cap)
final_validation_metrics = evaluate_model(validation_loader, maximum_frames=20, particle_cap=final_particle_cap)
final_testing_metrics = evaluate_model(testing_loader, maximum_frames=20, particle_cap=final_particle_cap)

final_metrics = {
    'training': final_training_metrics,
    'validation': final_validation_metrics,
    'testing': final_testing_metrics,
}
print(json.dumps(final_metrics, indent=2))

fig, axis = plt.subplots(figsize=(7.2, 4.5), constrained_layout=True)
split_names = list(final_metrics.keys())
values = [final_metrics[name]['relative_error'] for name in split_names]
bars = axis.bar(split_names, values, color=['#4c78a8', '#59a14f', '#e15759'])
axis.set_ylabel('Relative error in physical units')
axis.set_title('Final error by split')
axis.grid(axis='y', alpha=0.25)
for bar, value in zip(bars, values):
    axis.text(bar.get_x() + bar.get_width() / 2, value, f'{value:.3f}', ha='center', va='bottom')
plot_path = results_folder / 'final_relative_error_by_split.png'
fig.savefig(plot_path, dpi=220, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)


In [ ]:
# One readable prediction plot.
# Coordinates are input particle locations. The model is predicting velocity and velocity gradient values on those particles.

@torch.no_grad()
def predict_one_frame(dataset, dataset_index=0, maximum_particles_for_plot=12000):
    model.eval()
    input_tensor, target_tensor, raw_input_tensor, metadata = dataset[dataset_index]
    input_tensor, target_tensor, raw_input_tensor = subsample_before_gpu(
        input_tensor, target_tensor, raw_input_tensor, maximum_particles_for_plot
    )
    prediction_normalized = model(input_tensor.to(device)).float().cpu()
    target_physical = denormalize_target(target_tensor.to(device)).cpu().numpy()
    prediction_physical = denormalize_target(prediction_normalized.to(device)).cpu().numpy()
    raw_input = raw_input_tensor.numpy()
    return raw_input[:, :3], target_physical, prediction_physical, metadata


def robust_color_limits(values, lower=0.02, upper=0.98):
    finite_values = np.asarray(values)[np.isfinite(values)]
    if finite_values.size == 0:
        return 0.0, 1.0
    low = float(np.quantile(finite_values, lower))
    high = float(np.quantile(finite_values, upper))
    if low >= high:
        high = low + 1e-6
    return low, high

coordinates, true_values, predicted_values, metadata = predict_one_frame(validation_dataset, dataset_index=0)
true_velocity = np.linalg.norm(true_values[:, :3], axis=1)
predicted_velocity = np.linalg.norm(predicted_values[:, :3], axis=1)
velocity_error = predicted_velocity - true_velocity
true_gradient = np.linalg.norm(true_values[:, 3:], axis=1)
predicted_gradient = np.linalg.norm(predicted_values[:, 3:], axis=1)
gradient_error = predicted_gradient - true_gradient

figure, axes = plt.subplots(2, 3, figsize=(15, 8.5), constrained_layout=True)

velocity_low, velocity_high = robust_color_limits(np.concatenate([true_velocity, predicted_velocity]))
gradient_low, gradient_high = robust_color_limits(np.concatenate([true_gradient, predicted_gradient]))
velocity_error_limit = max(abs(robust_color_limits(velocity_error)[0]), abs(robust_color_limits(velocity_error)[1]))
gradient_error_limit = max(abs(robust_color_limits(gradient_error)[0]), abs(robust_color_limits(gradient_error)[1]))

plot_items = [
    (0, 0, true_velocity, '|u| true', 'viridis', velocity_low, velocity_high),
    (0, 1, predicted_velocity, '|u| prediction', 'viridis', velocity_low, velocity_high),
    (0, 2, velocity_error, '|u| signed error', 'RdBu_r', -velocity_error_limit, velocity_error_limit),
    (1, 0, true_gradient, '|gradU| true', 'magma', gradient_low, gradient_high),
    (1, 1, predicted_gradient, '|gradU| prediction', 'magma', gradient_low, gradient_high),
    (1, 2, gradient_error, '|gradU| signed error', 'RdBu_r', -gradient_error_limit, gradient_error_limit),
]

for row, column, values, title, color_map, low, high in plot_items:
    scatter = axes[row, column].scatter(
        coordinates[:, 0], coordinates[:, 2], c=values, s=3, cmap=color_map, vmin=low, vmax=high
    )
    axes[row, column].set_title(title)
    axes[row, column].set_xlabel('x')
    axes[row, column].set_ylabel('z')
    axes[row, column].grid(alpha=0.18)
    plt.colorbar(scatter, ax=axes[row, column], fraction=0.046, pad=0.02)

figure.suptitle(
    f"Validation frame: case {metadata['case']}, frame {metadata['frame']}, particles shown {coordinates.shape[0]}",
    fontsize=13,
)
plot_path = results_folder / 'validation_prediction_example.png'
figure.savefig(plot_path, dpi=240, bbox_inches='tight')
plt.show()
print('Saved:', plot_path)


## What to do if the model is not learning

Use this order of checks:

1. Run `training_profile = 'quick_check'` first. The training loss should usually move within a few epochs.
2. If training loss moves but validation/testing error is bad, the model is learning the training distribution but not generalizing well.
3. If validation/testing target magnitudes are very different from training magnitudes in the data scale check, normalization helps optimization but cannot fully fix missing physical coverage.
4. For better generalization, regenerate or add cases so training covers the same ranges of angle of attack, freestream speed, particle strength, and geometry behavior that you expect at validation/testing time.
5. If the learning rate looks like zero in a plot, check the printed value. It is normally a small number like `3e-4`; it only looked zero before because it was plotted on the same axis as thousands of particles.
